# Реконструкция 4D-КТ Ника из уменьшенного разрешения

**Вопрос исследования.** Восстанавливает ли маршрут `WorkflowReconstructHighres4DCT` MONAI Physio целевую фазу 4D-КТ Ника точнее обычного увеличения изображения, если на вход подать уменьшенную целевую фазу и полноценную опорную фазу 00? Выполнена ретроспективная проверка на двух реально снятых фазах. По заранее выбранной средней абсолютной ошибке в области сердца реконструкция уступила линейной интерполяции.

[Описание учебного маршрута MONAI Physio](https://project-monai.github.io/monai-physio/tutorials.html#tutorial-3-reconstruct-high-resolution-4d-ct) задаёт способ обработки изображений. Название «высокое разрешение» относится к выходной сетке и не означает, что алгоритм создал измеренную анатомическую информацию.

## Исходные данные и разрешение

Использованы 4D-КТ Ника из DICOM-серии 307: опорная фаза 00 и целевые фазы 01 и 07. Их исходная NIfTI-сетка имеет размер 512 × 512 × 256 и шаг 0,765625 × 0,765625 × 0,625 мм по аффинной матрице. Заголовок NIfTI не задаёт единицу длины; миллиметры подтверждаются происхождением из DICOM и конверсионным манифестом. Фазы имеют согласованные аффинные матрицы. Независимой второй съёмки более высокого разрешения того же состояния нет.

Область обработки задана *только* автоматической маской всего сердца фазы 00 TotalSegmentator v2 с запасом 25 мм. Маска целевой фазы в обработке не использовалась. Автоматический контур служит для выбора области вычисления и оценки, но не является принятой вручную анатомической истиной. Хэши конкретных входных файлов, размер обрезанной области и версии среды записаны в пофазных [JSON для 01](mp08_nix_real_holdout_phase_01.json) и [JSON для 07](mp08_nix_real_holdout_phase_07.json).

## Разделение входа и удержанного контроля

Для каждой целевой фазы исходное изображение сглажено гауссовым фильтром с σ = 0,8 вокселя и уменьшено вдвое по каждой оси. Шаг полученного входа составил 1,53125 × 1,53125 × 1,25 мм. На вход реконструктора переданы это уменьшенное изображение и реальная опорная КТ фазы 00 в полном исходном разрешении. После создания уменьшенной копии исходное изображение *целевой* фазы не подавали в реконструктор: его удержали для оценки результата; целевая маска также не использовалась.

Для сравнения то же уменьшенное изображение целевой фазы увеличено обычной линейной интерполяцией. Сохранённая опорная фаза 00 также оценена как неподвижный ориентир. Все три изображения перенесены на одну исходную сетку целевой фазы. Тем самым различия метрик относятся к способу восстановления при одинаковой низкоразрешающей целевой информации, а не к выбору разных контрольных вокселей.

## Алгоритм и проверка изображения

Применён `WorkflowReconstructHighres4DCT.process` с деформируемой регистрацией Greedy по взаимной корреляции (CC): итерации [12, 6, 3], сглаживание `3.0vox 1.0vox`, режим композиции `reference`. При работе с ITK геометрия NIfTI переведена из RAS в LPS. Размер собственной выходной сетки маршрута отличается от сетки исходного тома; перед оценкой и результат MONAI, и линейный контроль перенесены на исходную сетку.

Основная метрика — средняя абсолютная ошибка интенсивности (MAE) в единицах Хаунсфилда, HU, между восстановленным и удержанным исходным изображением. Она рассчитана в 1 703 924 вокселях, определённых маской `whole_heart` **опорной фазы 00**. Меньшая MAE лучше. Дополнительная среднеквадратичная ошибка (RMSE) сильнее реагирует на большие локальные отклонения. Эти метрики измеряют сходство изображений в данной области; они не устанавливают анатомическую правильность границ сердца.

## Результаты на двух удержанных фазах

| Целевая фаза | MAE неподвижной фазы 00, HU | MAE линейной интерполяции, HU | MAE MONAI Physio, HU | Отношение MAE MONAI к контролю | RMSE: интерполяция → MONAI, HU |
|---|---:|---:|---:|---:|---:|
| 01 | 26,58 | **13,07** | 25,26 | 1,93 | 31,58 → 45,02 |
| 07 | 27,41 | **12,14** | 21,74 | 1,79 | 29,65 → 29,32 |

В фазах 01 и 07 MAE маршрута MONAI выше MAE простой интерполяции соответственно на **12,19 и 9,60 HU**. Для фазы 07 дополнительная RMSE почти одинакова и немного ниже у MONAI; это не меняет отрицательной оценки по заранее выбранной MAE. Визуальная резкость выхода не показывает, совпадают ли отдельные детали с удержанной КТ. Сохранённые реконструированные NIfTI были повторно прочитаны; пересчитанная MAE совпала с JSON с отклонением менее 0,001 HU.

Это опыт с *реально измеренной* анатомией Ника и искусственным понижением её разрешения. Отдельной независимой высокоразрешающей съёмки нет. Поэтому результат относится к восстановлению известного оригинала из уменьшенной копии, а не к истинной сверхразрешающей способности при повторном сканировании.

## Ограничения, предшествующие проверки и решение

Вывод об отсутствии улучшения относится к двум фазам, выбранному режиму `reference`, заданным параметрам регистрации и MAE в области автоматической маски фазы 00. Другие режимы и состояния не сравнивались. Параметры во время этого опыта не подгоняли, но взяли из предыдущей ветки регистрации, где фазу 07 уже просматривали; выбор параметров нельзя считать полностью слепым. Анатомическая точность, влияние на сегментацию и функции сердца не проверялись. В текущей постановке MP-08 нельзя принять как способ улучшения КТ для основного анализа.

Предварительная проверка заголовков Ника показала согласованную сетку и пригодность файлов к обработке; её прежний статус «реконструкция ещё не выполнена» относится к состоянию до этого опыта. Более ранний просмотр заголовков Georg касался другого человека: при сетке 512 × 512 × 256 его шаг был 0,458984 × 0,458984 × 0,625 мм, а номинальные метки фаз 23–164 % не образуют автоматически один механический цикл. Эти данные в численный опыт Ника не входили.

Отдельный синтетический тест интерфейса использовал два фантома 32³ с известным сдвигом (2, −1, 0) мм и опорой 48³. Программа вернула объёмы 43³ с шагом 0,75 мм, но при коротком расписании оптимизации [8, 4, 2] ошибка восстановления сдвига в центре менялась примерно от 1,05 до 1,20 мм. Это подтверждает исполнимость интерфейса, а не точность на пациенте. Подробный [протокол на КТ Ника](mp08_nix_real_holdout_report.md) и [исполняемый скрипт](mp08_nix_real_holdout.py) позволяют повторить расчёт; индивидуальные тома и листы сравнения хранятся локально в исключённом из Git `runs/`.